In [ ]:
# %% [markdown]
# # 🧹 BrightCart Data Cleaning & Preparation
# 
# This notebook simulates the Python side of a real client project:
# cleaning, merging, and preparing e-commerce data for use in a Power BI dashboard.
# 
# **Goal:** produce a clean dataset with key marketing and sales metrics:
# - Revenue
# - Net Sales
# - Profit
# - ROAS (Return on Ad Spend)
# - CPC (Cost per Click)
# - Conversion Rate (%)

# %% [markdown]
# ## 1️⃣ Import Libraries

import pandas as pd
import numpy as np
import os

# Set working directory to project root (adjust if needed)
os.chdir("..")

# %% [markdown]
# ## 2️⃣ Load Raw Data

sales = pd.read_csv("data/raw/sales_raw.csv")
ads = pd.read_csv("data/raw/ad_spend.csv")
sessions = pd.read_csv("data/raw/website_sessions.csv")

print("✅ Data loaded successfully")
print("Sales shape:", sales.shape)
print("Ad spend shape:", ads.shape)
print("Sessions shape:", sessions.shape)

# %% [markdown]
# ## 3️⃣ Inspect the First Few Rows

print("\n--- Sales Data ---")
display(sales.head())

print("\n--- Ad Spend ---")
display(ads.head())

print("\n--- Website Sessions ---")
display(sessions.head())

# %% [markdown]
# ## 4️⃣ Basic Cleaning
# - Ensure consistent date types
# - Remove duplicates
# - Handle missing values

sales["Date"] = pd.to_datetime(sales["Date"])
ads["Week"] = pd.to_datetime(ads["Week"])
sessions["Week"] = pd.to_datetime(sessions["Week"])

sales.drop_duplicates(inplace=True)
ads.drop_duplicates(inplace=True)
sessions.drop_duplicates(inplace=True)

print("✅ Basic cleaning complete")

# %% [markdown]
# ## 5️⃣ Add Calculated Fields to Sales

sales["Revenue"] = sales["Units_Sold"] * sales["Unit_Price"]
sales["Net_Sales"] = sales["Revenue"] - sales["Discount"]

print("✅ Sales metrics added")
display(sales.head())

# %% [markdown]
# ## 6️⃣ Aggregate Weekly Sales by Channel

weekly_sales = (
    sales.groupby(["Date", "Channel"], as_index=False)
    .agg({"Units_Sold": "sum", "Revenue": "sum", "Net_Sales": "sum"})
    .rename(columns={"Date": "Week"})
)

print("✅ Aggregated weekly sales")
display(weekly_sales.head())

# %% [markdown]
# ## 7️⃣ Merge with Ad Spend and Website Sessions

merged = (
    weekly_sales
    .merge(ads, on=["Week", "Channel"], how="left")
    .merge(sessions, left_on=["Week", "Channel"], right_on=["Week", "Source"], how="left")
    .drop(columns=["Source"])
)

print("✅ Data merged successfully")
display(merged.head())

# %% [markdown]
# ## 8️⃣ Add KPI Calculations

merged["Profit"] = merged["Net_Sales"] - merged["Spend"]
merged["ROAS"] = (merged["Net_Sales"] / merged["Spend"]).round(2)
merged["CPC"] = (merged["Spend"] / merged["Clicks"]).round(2)
merged["Conversion_Rate"] = (merged["Conversions"] / merged["Sessions"] * 100).round(2)
merged["Profit_Margin"] = (merged["Profit"] / merged["Net_Sales"] * 100).round(2)

print("✅ KPIs calculated")
display(merged.head())

# %% [markdown]
# ## 9️⃣ Export Clean Dataset

os.makedirs("data/cleaned", exist_ok=True)
merged.to_csv("data/cleaned/brightcart_clean.csv", index=False)

print("✅ Clean dataset exported to data/cleaned/brightcart_clean.csv")

# %% [markdown]
# ## 🔚 Notebook Complete
# 
# Next step: open Power BI → Get Data → Text/CSV → 
# select `data/cleaned/brightcart_clean.csv` and build your dashboard.


✅ Data loaded successfully
Sales shape: (881, 7)
Ad spend shape: (104, 5)
Sessions shape: (104, 4)

--- Sales Data ---


,Date,Product_ID,Category,Units_Sold,Unit_Price,Discount,Channel
0,2025-01-05,P115,Home Decor,25,31.381957,0,Email
1,2025-01-05,P111,Home Decor,40,30.001016,5,Meta Ads
2,2025-01-05,P102,Home Decor,34,37.295607,0,Email
3,2025-01-05,P101,Home Decor,26,15.741962,0,Meta Ads
4,2025-01-05,P110,Home Decor,20,39.440991,0,Google Ads



--- Ad Spend ---


,Week,Channel,Spend,Clicks,Impressions
0,2025-01-05,Google Ads,752.772845,445,8010
1,2025-01-05,Meta Ads,1745.464565,325,4225
2,2025-01-05,Email,1465.426946,165,1650
3,2025-01-05,Organic Search,1845.694267,263,4471
4,2025-01-12,Google Ads,1485.198010,485,11640



--- Website Sessions ---


...

✅ Data merged successfully


,Week,Channel,Units_Sold,Revenue,Net_Sales,Spend,Clicks,Impressions,Sessions,Conversions
0,2025-01-05,Email,252,16344.981098,16324.981098,1465.426946,165,1650,3454,206
1,2025-01-05,Google Ads,299,21256.783144,21226.783144,752.772845,445,8010,3915,47
2,2025-01-05,Meta Ads,271,13979.245079,13944.245079,1745.464565,325,4225,3445,115
3,2025-01-05,Organic Search,75,3983.755656,3973.755656,1845.694267,263,4471,2365,103
4,2025-01-12,Email,64,4086.346544,4071.346544,2111.957802,200,4200,954,25


✅ KPIs calculated


,Week,Channel,Units_Sold,Revenue,Net_Sales,Spend,Clicks,Impressions,Sessions,Conversions,Profit,ROAS,CPC,Conversion_Rate,Profit_Margin
0,2025-01-05,Email,252,16344.981098,16324.981098,1465.426946,165,1650,3454,206,14859.554152,11.14,8.88,5.96,91.02
1,2025-01-05,Google Ads,299,21256.783144,21226.783144,752.772845,445,8010,3915,47,20474.010299,28.20,1.69,1.20,96.45
2,2025-01-05,Meta Ads,271,13979.245079,13944.245079,1745.464565,325,4225,3445,115,12198.780514,7.99,5.37,3.34,87.48
3,2025-01-05,Organic Search,75,3983.755656,3973.755656,1845.694267,263,4471,2365,103,2128.061389,2.15,7.02,4.36,53.55
4,2025-01-12,Email,64,4086.346544,4071.346544,2111.957802,200,4200,954,25,1959.388743,1.93,10.56,2.62,48.13


✅ Clean dataset exported to data/cleaned/brightcart_clean.csv
